# 21. Read segmentation polygons

Run image inference and read the polygons returned by MVI

Before running: Set MODEL_ID to a deployed segmentation model and IMAGE_PATH to a local image

Fill in `API_URL` and `API_KEY` in the next cell; set other values if needed

In [ ]:
import requests
from PIL import Image, ImageDraw
from IPython.display import display

API_URL = ""
API_KEY = ""
VERIFY_SSL = False # Set to True if you want to verify SSL certificates
MODEL_ID = '' # The model_id can be found in the URL for that model in the MVI UI
IMAGE_PATH = './example.jpg'

headers = {"X-Auth-Token": API_KEY}

In [ ]:
# Send the image to MVI and request polygon data
with open(IMAGE_PATH, "rb") as image_file:
    response = requests.post(
        f"{API_URL.rstrip('/')}/dlapis/{MODEL_ID}",
        headers=headers,
        files={"files": ("image", image_file)},
        data={"containPolygon": "true"},
        verify=VERIFY_SSL,
        timeout=120,
    )

response.raise_for_status()
result = response.json()

In [ ]:
# Read the labels and polygon coordinates from the API response
polygons = []

for item in result.get("classified", []):
    for polygon in item.get("polygons", []):
        points = [tuple(point) for point in polygon]
        if len(points) >= 3:
            polygons.append({"label": item.get("label"), "points": points})

print("Polygons:", len(polygons))
for polygon in polygons:
    print(polygon["label"], polygon["points"])

In [ ]:
# Draw the polygons on the original image
image = Image.open(IMAGE_PATH).convert("RGB")
draw = ImageDraw.Draw(image)

for polygon in polygons:
    points = polygon["points"]
    draw.line(points + [points[0]], fill="blue", width=3)

display(image)